# Alcatraz

Define bandages, replay moves with full colors, explore reachable shapes, and find a shortest shape solution.

Use the **v2/.venv** Python environment in VS Code's **Select Kernel** menu. Run the cells from top to bottom, then edit the bandage list or scramble and rerun the following cells.


In [ ]:
import sys
from pathlib import Path
from tempfile import TemporaryDirectory

import bce_v2 as c
import matplotlib.pyplot as plt

%matplotlib inline
print("Python kernel:", sys.executable)


## Define the puzzle

Equal positive labels fuse cells into one block. Each zero is an independent singleton. The named cell constants, such as `c.F` and `c.DF`, index this 27-cell grid.

Edit the list to experiment with other connected bandages. For the recorded fixture, you can also use `c.fixture("Alcatraz")`.


In [ ]:
alcatraz = [
      6, 6, 0,
     6, 6, 0,
    0, 0, 0,
      7, 7, 5,
     7, 7, 4,
    1, 2, 3,
      7, 7, 5,
     7, 7, 4,
    1, 2, 3,
]

initial = c.Shape(alcatraz)
initial.labels


In [ ]:
figure = c.draw_cubes(initial)
figure.axes[0].set_title("Reference shape")
plt.show()


## Replay a scramble

Change the move string below. All moves use standard Singmaster notation: `U R F D L B`, with optional `2` or `'`. A blocked move raises `c.BlockedMoveError`.

`State` retains corner and edge identities and orientations. Applying moves returns a new state.


In [ ]:
moves = "F R2"
scrambled = c.State(initial).apply(moves)
c.draw_cubes(scrambled)
plt.show()

{
    "scramble": scrambled.scramble,
    "legal_moves": scrambled.legal_moves,
    "colored_solved": scrambled.is_solved,
    "corners": scrambled.corners,
    "twists": scrambled.twists,
    "edges": scrambled.edges,
    "flips": scrambled.flips,
}


## Explore the shape component

Exploration runs to completion by default. The graph retains distinct move actions and self-loops. The recorded Alcatraz puzzle has 1,449 shapes and 2,048 clockwise arcs in QTM.


In [ ]:
graph = c.explore(initial, metric="QTM")
assert graph.complete

{
    "shapes": len(graph),
    "clockwise_arcs": len(graph.arcs),
    "metric": graph.metric,
    "complete": graph.complete,
}


## Restore the bandage shape

A shortest shape solution restores geometry. Whether the colors are solved is a separate question, checked by `State.is_solved`. Colored solving is planned for milestone four.


In [ ]:
solution = graph.shortest_path(scrambled, initial)
restored = scrambled.apply(solution)
assert restored.shape == initial

{
    "shape_solution": solution,
    "quarter_turns": graph.distances(initial)[graph.vertex_id(scrambled)],
    "shape_restored": restored.shape == initial,
    "colors_solved": restored.is_solved,
}


In [ ]:
figure = c.draw_cubes([initial, scrambled, restored], alpha=0.35)
for axis, title in zip(figure.axes, ["Reference", moves, f"After {solution or 'no moves'}"]):
    axis.set_title(title)
plt.show()


## Compare move metrics

QTM counts half turns as two quarter turns. HTM counts every face turn, including a half turn, as one move. Pass shapes or states between graphs; vertex IDs belong to an individual exploration.


In [ ]:
htm_graph = c.explore(initial, metric="HTM")
htm_solution = htm_graph.shortest_path(scrambled, initial)
assert scrambled.apply(htm_solution).shape == initial

{
    "QTM": {
        "solution": solution,
        "cost": graph.distances(initial)[graph.vertex_id(scrambled)],
    },
    "HTM": {
        "solution": htm_solution,
        "cost": htm_graph.distances(initial)[htm_graph.vertex_id(scrambled)],
    },
}


## Inspect interesting shapes

Filter shapes using ordinary Python expressions. This selection finds shapes that allow front turns; change the expression to inspect something else. Draw a small selection for a gallery.


In [ ]:
free_front = [value for value in graph if value.is_turnable("F")]
print(f"{len(free_front)} shapes permit front turns")

figure = c.draw_cubes(free_front[:6], ncol=3, size=3)
plt.show()


## Save a reproducible experiment

Puzzle records retain reference bandages and the legal scramble needed to reproduce full colors. Graph exports retain their metric, completeness status, normalized shapes, and move actions.

This example uses temporary files. To keep results, call `c.save_puzzle(scrambled, "experiment.json")` and `graph.save("shapes.json")` with your own paths.


In [ ]:
with TemporaryDirectory() as directory:
    puzzle_path = Path(directory) / "Alcatraz.json"
    graph_path = Path(directory) / "Alcatraz-shapes.json"

    record = c.save_puzzle(scrambled, puzzle_path, name="Alcatraz experiment")
    graph.save(graph_path)

    loaded = c.load_puzzle(puzzle_path)
    assert loaded == scrambled
    graph_export_bytes = graph_path.stat().st_size

{
    "colored_round_trip": loaded == scrambled,
    "recorded_scramble": record["scramble"],
    "graph_export_bytes": graph_export_bytes,
}
